# Investment Problem in Pure Python
## Five Exact Methods for Book Problem 2.8

This notebook presents five exact approaches for the investment problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve:
- the base investment model from book section `2.8`
- the aggregated student variation with total purchases by artifact type
- the student variation with individual budgets per traveler

Each method reports:
- one optimal purchase plan
- the maximum profit
- the execution time


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


In [2]:
SCALE = 10000


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def profit_from_values(values, profits):
    return sum(value * profit for value, profit in zip(values, profits))


def solution_from_values(problem, values):
    result = {
        name: int(value)
        for name, value in zip(problem["variable_names"], values)
    }
    result["profit"] = int(profit_from_values(values, problem["profits"]))
    return result


def is_feasible(problem, values):
    total_cost = sum(value * cost for value, cost in zip(values, problem["costs"]))
    total_items = sum(values)
    return total_cost <= problem["budget"] and total_items <= problem["max_items"]


def choose_better(problem, left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_profit = profit_from_values(left, problem["profits"])
    right_profit = profit_from_values(right, problem["profits"])
    if right_profit > left_profit:
        return right
    if right_profit < left_profit:
        return left
    return right if tuple(right) < tuple(left) else left


In [3]:
PROFITS = [150000, 90000, 60000]
COSTS = [250000, 180000, 70000]
NAMES = ["televisions", "sound_systems", "vacuums"]

BASE_PROBLEM = {
    "profits": PROFITS,
    "costs": COSTS,
    "budget": 3000000,
    "max_items": 48,
    "variable_names": NAMES,
}

VARIANT_1_PROBLEM = {
    "profits": PROFITS,
    "costs": COSTS,
    "budget": 3000000,
    "max_items": 48,
    "variable_names": NAMES,
}

VARIANT_2_BUNDLE = {
    "person_1": {"profits": PROFITS, "costs": COSTS, "budget": 3000000, "max_items": 12, "variable_names": NAMES},
    "person_2": {"profits": PROFITS, "costs": COSTS, "budget": 4000000, "max_items": 12, "variable_names": NAMES},
    "person_3": {"profits": PROFITS, "costs": COSTS, "budget": 2500000, "max_items": 12, "variable_names": NAMES},
    "person_4": {"profits": PROFITS, "costs": COSTS, "budget": 1900000, "max_items": 12, "variable_names": NAMES},
}

BASE_EXPECTED = {
    "televisions": 0,
    "sound_systems": 0,
    "vacuums": 42,
    "profit": 2520000,
}


## Problem 1: Base Investment Model

The book formulation distinguishes travelers, but the base problem has:
- one global budget of `3,000,000 COP`
- at most `12` items per traveler
- identical profit and cost data for every traveler

That means the exact search can be performed on total purchases by artifact type, with an aggregate cap of `48` items.


In [4]:
def solve_three_item_naive(problem):
    max_t = min(problem["max_items"], problem["budget"] // problem["costs"][0])
    max_s = min(problem["max_items"], problem["budget"] // problem["costs"][1])
    max_v = min(problem["max_items"], problem["budget"] // problem["costs"][2])
    best = None

    for televisions in range(max_t + 1):
        for sound_systems in range(max_s + 1):
            for vacuums in range(max_v + 1):
                values = (televisions, sound_systems, vacuums)
                if not is_feasible(problem, values):
                    continue
                best = choose_better(problem, best, values)

    return solution_from_values(problem, best)


def solve_three_item_backtracking(problem):
    best = None
    max_profit_per_item = max(problem["profits"])

    def backtrack(index, current_values, used_cost, used_items, current_profit):
        nonlocal best

        if index == 3:
            best = choose_better(problem, best, tuple(current_values))
            return

        remaining_budget = problem["budget"] - used_cost
        remaining_items = problem["max_items"] - used_items
        optimistic_profit = current_profit + remaining_items * max_profit_per_item
        if best is not None and optimistic_profit < profit_from_values(best, problem["profits"]):
            return

        max_value = min(
            remaining_items,
            remaining_budget // problem["costs"][index],
        )
        for value in range(max_value, -1, -1):
            current_values[index] = value
            backtrack(
                index + 1,
                current_values,
                used_cost + value * problem["costs"][index],
                used_items + value,
                current_profit + value * problem["profits"][index],
            )
        current_values[index] = 0

    backtrack(0, [0, 0, 0], 0, 0, 0)
    return solution_from_values(problem, best)


def solve_three_item_reduced_search(problem):
    best = None
    max_t = min(problem["max_items"], problem["budget"] // problem["costs"][0])
    max_s = min(problem["max_items"], problem["budget"] // problem["costs"][1])

    for televisions in range(max_t + 1):
        for sound_systems in range(max_s + 1):
            used_cost = televisions * problem["costs"][0] + sound_systems * problem["costs"][1]
            used_items = televisions + sound_systems
            if used_cost > problem["budget"] or used_items > problem["max_items"]:
                continue
            vacuums = min(
                problem["max_items"] - used_items,
                (problem["budget"] - used_cost) // problem["costs"][2],
            )
            values = (televisions, sound_systems, vacuums)
            best = choose_better(problem, best, values)

    return solution_from_values(problem, best)


def solve_three_item_dp(problem):
    scaled_budget = problem["budget"] // SCALE
    scaled_costs = [cost // SCALE for cost in problem["costs"]]
    scaled_profits = [profit // SCALE for profit in problem["profits"]]

    @lru_cache(maxsize=None)
    def dp(index, remaining_budget, remaining_items):
        if index == 3:
            return 0, ()

        best_profit = -1
        best_suffix = None
        max_value = min(remaining_items, remaining_budget // scaled_costs[index])
        for value in range(max_value + 1):
            suffix_profit, suffix_values = dp(
                index + 1,
                remaining_budget - value * scaled_costs[index],
                remaining_items - value,
            )
            total_profit = value * scaled_profits[index] + suffix_profit
            candidate = (value,) + suffix_values
            if total_profit > best_profit:
                best_profit = total_profit
                best_suffix = candidate
            elif total_profit == best_profit and candidate < best_suffix:
                best_suffix = candidate
        return best_profit, best_suffix

    _, values = dp(0, scaled_budget, problem["max_items"])
    return solution_from_values(problem, values)


def solve_three_item_branch_and_bound(problem):
    best = None
    density_order = [2, 0, 1]

    def fractional_upper_bound(index, remaining_budget, remaining_items, current_profit):
        bound = current_profit
        budget_left = remaining_budget
        items_left = remaining_items

        for product in density_order:
            if product < index:
                continue
            if items_left == 0:
                break
            max_by_budget = budget_left / problem["costs"][product]
            take = min(items_left, max_by_budget)
            bound += take * problem["profits"][product]
            budget_left -= take * problem["costs"][product]
            items_left -= take
        return bound

    stack = [(0, (0, 0, 0), problem["budget"], problem["max_items"], 0)]
    while stack:
        index, current_values, remaining_budget, remaining_items, current_profit = stack.pop()
        if index == 3:
            best = choose_better(problem, best, current_values)
            continue

        if best is not None and fractional_upper_bound(index, remaining_budget, remaining_items, current_profit) < profit_from_values(best, problem["profits"]):
            continue

        max_value = min(remaining_items, remaining_budget // problem["costs"][index])
        for value in range(max_value, -1, -1):
            next_values = list(current_values)
            next_values[index] = value
            stack.append(
                (
                    index + 1,
                    tuple(next_values),
                    remaining_budget - value * problem["costs"][index],
                    remaining_items - value,
                    current_profit + value * problem["profits"][index],
                )
            )

    return solution_from_values(problem, best)


def solve_bundle(bundle, solver):
    results = {}
    total_profit = 0
    for name, problem in bundle.items():
        solution = solver(problem)
        results[name] = solution
        total_profit += solution["profit"]
    results["total_profit"] = total_profit
    return results


In [5]:
@timed
def solve_base_naive():
    return solve_three_item_naive(BASE_PROBLEM)


@timed
def solve_base_backtracking():
    return solve_three_item_backtracking(BASE_PROBLEM)


@timed
def solve_base_reduced_search():
    return solve_three_item_reduced_search(BASE_PROBLEM)


@timed
def solve_base_dp():
    return solve_three_item_dp(BASE_PROBLEM)


@timed
def solve_base_branch_and_bound():
    return solve_three_item_branch_and_bound(BASE_PROBLEM)


In [6]:
base_naive_result, base_naive_time = solve_base_naive()
base_backtracking_result, base_backtracking_time = solve_base_backtracking()
base_reduced_result, base_reduced_time = solve_base_reduced_search()
base_dp_result, base_dp_time = solve_base_dp()
base_bb_result, base_bb_time = solve_base_branch_and_bound()

print("=== BASE INVESTMENT RESULTS ===")
print(f"Naive enumeration            -> {base_naive_result}, time = {base_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {base_backtracking_result}, time = {base_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {base_reduced_result}, time = {base_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {base_dp_result}, time = {base_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {base_bb_result}, time = {base_bb_time:.8f} seconds")

assert base_naive_result == BASE_EXPECTED
assert base_backtracking_result == base_naive_result
assert base_reduced_result == base_naive_result
assert base_dp_result == base_naive_result
assert base_bb_result == base_naive_result
print("All five exact methods agree with the textbook optimum.")


=== BASE INVESTMENT RESULTS ===
Naive enumeration            -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00500946 seconds
Backtracking with pruning    -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00179783 seconds
Constraint-driven reduced search -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00012558 seconds
Dynamic programming          -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00055800 seconds
Branch and Bound             -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00007908 seconds
All five exact methods agree with the textbook optimum.


## Problem 2: Aggregated Student Variation

The first student variation asks to define:
- `x_i`: total number of artifacts of type `i`

This is exactly the aggregated model already used in the reduced base search, so the optimum remains the same.


In [7]:
@timed
def solve_variant_1_naive():
    return solve_three_item_naive(VARIANT_1_PROBLEM)


@timed
def solve_variant_1_backtracking():
    return solve_three_item_backtracking(VARIANT_1_PROBLEM)


@timed
def solve_variant_1_reduced_search():
    return solve_three_item_reduced_search(VARIANT_1_PROBLEM)


@timed
def solve_variant_1_dp():
    return solve_three_item_dp(VARIANT_1_PROBLEM)


@timed
def solve_variant_1_branch_and_bound():
    return solve_three_item_branch_and_bound(VARIANT_1_PROBLEM)


In [8]:
variant_1_naive_result, variant_1_naive_time = solve_variant_1_naive()
variant_1_backtracking_result, variant_1_backtracking_time = solve_variant_1_backtracking()
variant_1_reduced_result, variant_1_reduced_time = solve_variant_1_reduced_search()
variant_1_dp_result, variant_1_dp_time = solve_variant_1_dp()
variant_1_bb_result, variant_1_bb_time = solve_variant_1_branch_and_bound()

print("=== AGGREGATED VARIATION RESULTS ===")
print(f"Naive enumeration            -> {variant_1_naive_result}, time = {variant_1_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {variant_1_backtracking_result}, time = {variant_1_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {variant_1_reduced_result}, time = {variant_1_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {variant_1_dp_result}, time = {variant_1_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {variant_1_bb_result}, time = {variant_1_bb_time:.8f} seconds")

assert variant_1_naive_result == BASE_EXPECTED
assert variant_1_backtracking_result == variant_1_naive_result
assert variant_1_reduced_result == variant_1_naive_result
assert variant_1_dp_result == variant_1_naive_result
assert variant_1_bb_result == variant_1_naive_result
print("The aggregated student variation keeps the same optimum.")


=== AGGREGATED VARIATION RESULTS ===
Naive enumeration            -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00495117 seconds
Backtracking with pruning    -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00163333 seconds
Constraint-driven reduced search -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00012200 seconds
Dynamic programming          -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00052229 seconds
Branch and Bound             -> {'televisions': 0, 'sound_systems': 0, 'vacuums': 42, 'profit': 2520000}, time = 0.00007904 seconds
The aggregated student variation keeps the same optimum.


## Problem 3: Student Variation with Individual Budgets

The second student variation gives each traveler a different budget:
- person 1: `3,000,000`
- person 2: `4,000,000`
- person 3: `2,500,000`
- person 4: `1,900,000`

The model separates exactly by traveler, so each method solves the same three-artifact problem four times and then aggregates the profits.


In [9]:
@timed
def solve_variant_2_naive():
    return solve_bundle(VARIANT_2_BUNDLE, solve_three_item_naive)


@timed
def solve_variant_2_backtracking():
    return solve_bundle(VARIANT_2_BUNDLE, solve_three_item_backtracking)


@timed
def solve_variant_2_reduced_search():
    return solve_bundle(VARIANT_2_BUNDLE, solve_three_item_reduced_search)


@timed
def solve_variant_2_dp():
    return solve_bundle(VARIANT_2_BUNDLE, solve_three_item_dp)


@timed
def solve_variant_2_branch_and_bound():
    return solve_bundle(VARIANT_2_BUNDLE, solve_three_item_branch_and_bound)


In [10]:
variant_2_naive_result, variant_2_naive_time = solve_variant_2_naive()
variant_2_backtracking_result, variant_2_backtracking_time = solve_variant_2_backtracking()
variant_2_reduced_result, variant_2_reduced_time = solve_variant_2_reduced_search()
variant_2_dp_result, variant_2_dp_time = solve_variant_2_dp()
variant_2_bb_result, variant_2_bb_time = solve_variant_2_branch_and_bound()

print("=== INDIVIDUAL-BUDGET VARIATION RESULTS ===")
print(f"Naive enumeration            -> {variant_2_naive_result}, time = {variant_2_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {variant_2_backtracking_result}, time = {variant_2_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {variant_2_reduced_result}, time = {variant_2_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {variant_2_dp_result}, time = {variant_2_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {variant_2_bb_result}, time = {variant_2_bb_time:.8f} seconds")

assert variant_2_naive_result["total_profit"] == 6330000
assert variant_2_backtracking_result == variant_2_naive_result
assert variant_2_reduced_result == variant_2_naive_result
assert variant_2_dp_result == variant_2_naive_result
assert variant_2_bb_result == variant_2_naive_result
print("All five exact methods agree on the separated-budget optimum.")


=== INDIVIDUAL-BUDGET VARIATION RESULTS ===
Naive enumeration            -> {'person_1': {'televisions': 12, 'sound_systems': 0, 'vacuums': 0, 'profit': 1800000}, 'person_2': {'televisions': 12, 'sound_systems': 0, 'vacuums': 0, 'profit': 1800000}, 'person_3': {'televisions': 9, 'sound_systems': 0, 'vacuums': 3, 'profit': 1530000}, 'person_4': {'televisions': 5, 'sound_systems': 1, 'vacuums': 6, 'profit': 1200000}, 'total_profit': 6330000}, time = 0.00425154 seconds
Backtracking with pruning    -> {'person_1': {'televisions': 12, 'sound_systems': 0, 'vacuums': 0, 'profit': 1800000}, 'person_2': {'televisions': 12, 'sound_systems': 0, 'vacuums': 0, 'profit': 1800000}, 'person_3': {'televisions': 9, 'sound_systems': 0, 'vacuums': 3, 'profit': 1530000}, 'person_4': {'televisions': 5, 'sound_systems': 1, 'vacuums': 6, 'profit': 1200000}, 'total_profit': 6330000}, time = 0.00105829 seconds
Constraint-driven reduced search -> {'person_1': {'televisions': 12, 'sound_systems': 0, 'vacuums': 0,